# Phase 0 / Task A1 — Data Exploration & Cleaning

One cell loads/inspects, the next cell cleans based on what we saw.  
All cleaning decisions are documented with reasoning inline.

Working directory: `c:\Users\typal\Downloads\mine-data-task-TP`

In [3]:
# --- Imports ---
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set display options so we can see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

# --- Find the repo root by walking up until we hit the .git folder ---
# This works on any machine regardless of where the repo is cloned.
def find_repo_root():
    path = Path().resolve()
    for p in [path] + list(path.parents):
        if (p / '.git').exists():
            return p
    return path  # fallback to cwd if .git not found

REPO_ROOT = find_repo_root()
CSV_DIR = str(REPO_ROOT / 'Data')    # CSVs are in the Data/ subfolder
DATA_DIR = str(REPO_ROOT)            # repo root (for saving outputs)

print('Repo root:', REPO_ROOT)
print('CSV dir:  ', CSV_DIR)
print('CSVs found:', [f for f in os.listdir(CSV_DIR) if f.endswith('.csv')])

Repo root: C:\Users\typal\Downloads\mine-data-task-TP
CSV dir:   C:\Users\typal\Downloads\mine-data-task-TP\Data
CSVs found: ['auto_rules.csv', 'buyer_actions.csv', 'campaign_adset_metadata.csv', 'daily_adset_performance.csv', 'rule_executions.csv']


---
## 1. `daily_adset_performance.csv`
Daily metrics per adset. Key concern: zero-spend rows, null revenue fields, and the two adset ID formats.

In [4]:
# --- Load and inspect ---
perf = pd.read_csv(os.path.join(CSV_DIR, 'daily_adset_performance.csv'))

print('Shape:', perf.shape)
print('\nDate range:', perf['date'].min(), '->', perf['date'].max())
print('Unique adsets:', perf['adset_id'].nunique())
print('Accounts:', sorted(perf['account_name'].unique()))
print('\nDtypes:')
print(perf.dtypes)
print('\nNull counts:')
print(perf.isnull().sum())

Shape: (4947, 18)

Date range: 2026-06-06 -> 2026-06-12
Unique adsets: 1000
Accounts: ['ACC-01', 'ACC-02', 'ACC-03', 'ACC-04', 'ACC-05', 'ACC-06']

Dtypes:
adset_id                   int64
fb_ad_account_id           int64
account_name              object
date                      object
spend                    float64
impressions                int64
clicks                     int64
fb_conversions             int64
estimated_conversions    float64
revenue                  float64
profit                   float64
roi                      float64
rpc                      float64
cpa                      float64
ctr                      float64
cr                       float64
first_spend_date          object
spend_day_no               int64
dtype: object

Null counts:
adset_id                    0
fb_ad_account_id            0
account_name                0
date                        0
spend                       0
impressions                 0
clicks                      0
fb_conversio

In [5]:
# --- Inspect zero-spend rows ---
# Zero-spend rows exist because paused adsets are still reported.
# We need to understand what fields go null when spend=0.
zero_spend = perf[perf['spend'] == 0]
nonzero_spend = perf[perf['spend'] > 0]

print(f'Zero-spend rows:    {len(zero_spend):,}  ({100*len(zero_spend)/len(perf):.1f}%)')
print(f'Non-zero-spend rows:{len(nonzero_spend):,}  ({100*len(nonzero_spend)/len(perf):.1f}%)')
print('\nNull counts in zero-spend rows (fields we care about):')
cols_to_check = ['revenue', 'profit', 'roi', 'estimated_conversions', 'rpc', 'cpa', 'ctr', 'cr']
print(zero_spend[cols_to_check].isnull().sum())
print('\nSample zero-spend row:')
print(zero_spend[cols_to_check].head(3))

Zero-spend rows:    3,000  (60.6%)
Non-zero-spend rows:1,947  (39.4%)

Null counts in zero-spend rows (fields we care about):
revenue                  2721
profit                   2721
roi                         0
estimated_conversions    2713
rpc                         0
cpa                         0
ctr                      2953
cr                       2992
dtype: int64

Sample zero-spend row:
    revenue  profit    roi  estimated_conversions    rpc    cpa  ctr  cr
15      NaN     NaN 0.0000                    NaN 0.0000 0.0000  NaN NaN
16      NaN     NaN 0.0000                    NaN 0.0000 0.0000  NaN NaN
17      NaN     NaN 0.0000                    NaN 0.0000 0.0000  NaN NaN


In [6]:
# --- Inspect adset ID formats ---
# README warns IDs appear in more than one format. Let's see the distribution.
perf['adset_id_str'] = perf['adset_id'].astype(str)
perf['id_len'] = perf['adset_id_str'].str.len()

print('Adset ID length distribution:')
print(perf.groupby('id_len')[['adset_id']].nunique().rename(columns={'adset_id': 'unique_adsets'}))

print('\nWhich accounts use which ID format?')
print(perf.groupby(['account_name', 'id_len'])['adset_id'].nunique().rename('unique_adsets').reset_index())

Adset ID length distribution:
        unique_adsets
id_len               
14                117
18                883

Which accounts use which ID format?
  account_name  id_len  unique_adsets
0       ACC-01      18             61
1       ACC-02      18            233
2       ACC-03      18            138
3       ACC-04      14            117
4       ACC-05      18            201
5       ACC-06      18            250


In [7]:
# --- CLEAN: daily_adset_performance ---
#
# Decision 1: Keep zero-spend rows but add a flag column.
#   Reason: zero-spend rows carry structural info (adset exists, was paused).
#   Dropping them would hide gaps in an adset's activity timeline.
#
# Decision 2: Fill nulls in ratio/metric cols with 0 for zero-spend rows only.
#   Reason: spend=0 => revenue=0, profit=0, roi=undefined. Using 0 is safe
#   for aggregation and avoids NA propagation. We flag them separately.
#
# Decision 3: Parse date column to datetime.
#   Reason: needed for all time-based joins and analysis.
#
# Decision 4: Keep adset_id as string to preserve both ID formats.
#   Reason: numeric parsing can silently lose leading/trailing precision on large ints.

perf_clean = perf.copy()

# Parse date
perf_clean['date'] = pd.to_datetime(perf_clean['date'])

# Flag zero-spend rows
perf_clean['is_zero_spend'] = perf_clean['spend'] == 0

# For zero-spend rows, fill metric nulls with 0
metric_cols = ['revenue', 'profit', 'roi', 'estimated_conversions',
               'rpc', 'cpa', 'ctr', 'cr', 'fb_conversions']
for col in metric_cols:
    if col in perf_clean.columns:
        perf_clean.loc[perf_clean['is_zero_spend'], col] = \
            perf_clean.loc[perf_clean['is_zero_spend'], col].fillna(0)

# Keep adset_id as string
perf_clean['adset_id'] = perf_clean['adset_id'].astype(str)

# Drop the helper cols we added during inspection
perf_clean = perf_clean.drop(columns=['adset_id_str', 'id_len'])

print('Cleaned shape:', perf_clean.shape)
print('Remaining nulls in metric cols:')
print(perf_clean[metric_cols].isnull().sum())
print('Zero-spend flag counts:', perf_clean['is_zero_spend'].value_counts().to_dict())

Cleaned shape: (4947, 19)
Remaining nulls in metric cols:
revenue                    0
profit                     0
roi                        0
estimated_conversions      0
rpc                        0
cpa                        0
ctr                        0
cr                       105
fb_conversions             0
dtype: int64
Zero-spend flag counts: {True: 3000, False: 1947}


---
## 2. `campaign_adset_metadata.csv`
Adset config: status, budgets, bid strategy, targeting. All IDs here appear to be long-format.

In [8]:
# --- Load and inspect ---
meta = pd.read_csv(os.path.join(CSV_DIR, 'campaign_adset_metadata.csv'))

print('Shape:', meta.shape)
print('Unique adsets:', meta['adset_id'].nunique())
print('Accounts:', sorted(meta['account_name'].unique()))
print('\nStatus breakdown:')
print(meta['effective_status'].value_counts())
print('\nDelivery status breakdown:')
print(meta['delivery_status'].value_counts())
print('\nNull counts:')
print(meta.isnull().sum())
print('\nDtypes:')
print(meta.dtypes)

Shape: (7129, 18)
Unique adsets: 7129
Accounts: ['ACC-01', 'ACC-02', 'ACC-03', 'ACC-04', 'ACC-05', 'ACC-06']

Status breakdown:
effective_status
ACTIVE     4023
PAUSED     3014
DELETED      92
Name: count, dtype: int64

Delivery status breakdown:
delivery_status
ADSET_OFF       3747
CAMPAIGN_OFF    2974
ACTIVE           305
DELETED           92
SCHEDULED         11
Name: count, dtype: int64

Null counts:
account_name              0
campaign_id               0
adset_id                  0
campaign_name             0
adset_name             1893
effective_status          0
delivery_status           0
daily_budget              0
bid_amount             6631
bid_strategy              0
budget_optimization       0
roas_target            3512
objective                 0
optimization_goal         0
geo_countries           755
language                484
creation_date             0
start_time                0
dtype: int64

Dtypes:
account_name            object
campaign_id              int64
adse

In [9]:
# --- Check ID format in metadata ---
meta['adset_id_str'] = meta['adset_id'].astype(str)
meta['id_len'] = meta['adset_id_str'].str.len()
print('ID length distribution in metadata:')
print(meta.groupby(['account_name', 'id_len'])['adset_id'].nunique())

# Check budget distribution
print('\nBudget (daily_budget) summary:')
print(meta['daily_budget'].describe())
print('\nBid strategy breakdown:')
print(meta['bid_strategy'].value_counts())

ID length distribution in metadata:
account_name  id_len
ACC-01        18         471
ACC-02        18         804
ACC-03        18         713
ACC-04        14         648
ACC-05        18         472
ACC-06        18        4021
Name: adset_id, dtype: int64

Budget (daily_budget) summary:
count    7129.0000
mean      398.7301
std       724.5605
min       127.0000
25%       127.0000
50%       254.0000
75%       508.0000
max     24130.0000
Name: daily_budget, dtype: float64

Bid strategy breakdown:
bid_strategy
LOWEST_COST_WITH_MIN_ROAS    3617
LOWEST_COST_WITHOUT_CAP      3014
LOWEST_COST_WITH_BID_CAP      452
COST_CAP                       46
Name: count, dtype: int64


In [10]:
# --- CLEAN: campaign_adset_metadata ---
#
# Decision 1: Keep adset_id as string.
#
# Decision 2: Parse creation_date and start_time to datetime.
#
# Decision 3: Keep nulls in bid_amount and roas_target — these are legitimately
#   optional fields (not all bid strategies require them).
#
# Decision 4: Drop helper inspection columns.

meta_clean = meta.copy()
meta_clean['adset_id'] = meta_clean['adset_id'].astype(str)
meta_clean['campaign_id'] = meta_clean['campaign_id'].astype(str)
meta_clean['creation_date'] = pd.to_datetime(meta_clean['creation_date'], utc=True, errors='coerce')
meta_clean['start_time'] = pd.to_datetime(meta_clean['start_time'], utc=True, errors='coerce')
meta_clean = meta_clean.drop(columns=['adset_id_str', 'id_len'])

print('Cleaned shape:', meta_clean.shape)
print('Remaining nulls:')
print(meta_clean.isnull().sum())

Cleaned shape: (7129, 18)
Remaining nulls:
account_name              0
campaign_id               0
adset_id                  0
campaign_name             0
adset_name             1893
effective_status          0
delivery_status           0
daily_budget              0
bid_amount             6631
bid_strategy              0
budget_optimization       0
roas_target            3512
objective                 0
optimization_goal         0
geo_countries           755
language                484
creation_date             0
start_time                0
dtype: int64


---
## 3. `auto_rules.csv`
Small table — 12 rules. Mainly need to confirm rule IDs match rule_executions.

In [11]:
# --- Load and display (small table, show it all) ---
rules = pd.read_csv(os.path.join(CSV_DIR, 'auto_rules.csv'))

print('Shape:', rules.shape)
print('\nAll rules:')
print(rules.to_string())
print('\nNull counts:')
print(rules.isnull().sum())
print('\nTotal firings this week:', rules['firings'].sum())

Shape: (12, 6)

All rules:
   rule_id                                                                             rule_name                 action          schedule                          scope  firings
0      R01                                                 Turn OFF | Total Days >= 5 | OWN RSOC               Turn OFF  every 30 minutes  OWN RSOC adsets, all accounts        4
1      R02                                         Budget Decrease | OWN RSOC | -30 < ROI <= -10  Decrease -20% (min 9)  every 30 minutes  OWN RSOC adsets, all accounts       23
2      R03                              Turn Off | positive_days = 0 | total_days > 2 | OWN RSOC               Turn OFF  every 30 minutes  OWN RSOC adsets, all accounts       36
3      R04                       Turn Off - OWN RSOC | Total Days = 1 | budget > 35%| ROI < -50%               Turn OFF  every 30 minutes  OWN RSOC adsets, all accounts      109
4      R05               Turn Off | OWN RSOC | Total Profit <= -2.5$ | budget_usage

In [12]:
# --- CLEAN: auto_rules ---
# No cleaning needed — small, complete table.
# Just store it as-is for reference joins.
rules_clean = rules.copy()
print('Rules clean — no changes made.')
print(rules_clean[['rule_id', 'rule_name', 'action', 'firings']])

Rules clean — no changes made.
   rule_id                                                    rule_name  \
0      R01                        Turn OFF | Total Days >= 5 | OWN RSOC   
1      R02                Budget Decrease | OWN RSOC | -30 < ROI <= -10   
2      R03     Turn Off | positive_days = 0 | total_days > 2 | OWN RSOC   
3      R04  Turn Off - OWN RSOC | Total Days = 1 | budget > 35%| ROI...   
4      R05  Turn Off | OWN RSOC | Total Profit <= -2.5$ | budget_usa...   
5      R06  Turn Off | today_profit <=  -1.25 $| total_days <= 3 | t...   
6      R07       Budget Decrease | OWN RSOC | -50 <= ROI | Budget > 65$   
7      R08                         Turn OFF | Total Days = 4 | OWN RSOC   
8      R09              Turn On | Automation Mistake - Today | OWN RSOC   
9      R10  Budget Decrease | OWN RSOC | -10 < ROI <= 5 | Budget >= ...   
10     R11     Turn Off | positive_days = 0 | total_days > 3 | OWN RSOC   
11     R12      Budget Decrease | OWN RSOC | ROI <= -50 | Budget <= 6

---
## 4. `rule_executions.csv`
Full log of every rule action. Key concerns: failed API calls (OAuthException), duplicate firings on same adset within minutes, and null budget fields.

In [13]:
# --- Load and inspect ---
execs = pd.read_csv(os.path.join(CSV_DIR, 'rule_executions.csv'))

print('Shape:', execs.shape)
print('Date range:', execs['action_date'].min(), '->', execs['action_date'].max())
print('Unique adsets:', execs['adset_id'].nunique())
print('\nFirings by rule_id:')
print(execs.groupby('rule_id').size().rename('firings').to_string())
print('\nNull counts:')
print(execs.isnull().sum())
print('\nUnique API response types (first 10):')
print(execs['response'].value_counts().head(10))

Shape: (214, 27)
Date range: 2026-06-06 -> 2026-06-12
Unique adsets: 75

Firings by rule_id:
rule_id
R01      4
R02     23
R03     36
R04    109
R05      7
R06      1
R07      1
R08     15
R09      8
R10      2
R11      2
R12      6

Null counts:
action_date                          0
action_time                          0
rule_name                            0
condition_name                       0
action_name                          0
account_id                           0
adset_id                             0
campaign_id                          0
old_budget                           2
new_budget                           2
set_budget                         197
response                             0
spend_at_action                      0
total_spend_at_action                0
total_days_at_action                 0
today_roi_at_action                  0
current_budget_from_fb              45
last_3_days_revenue_at_action       68
last_3_days_total_roi_at_action      0
today_rpc_at

In [14]:
# --- Inspect failed API calls ---
# Rows where the rule fired but the API returned an error = no actual impact.
# These must NOT be counted as real decisions in our impact analysis.

# A row is a success if response == 'SUCCESS'
execs['api_success'] = execs['response'].str.strip().str.upper() == 'SUCCESS'

print('API success breakdown:')
print(execs['api_success'].value_counts())
print('\nFailed response samples:')
print(execs[~execs['api_success']]['response'].value_counts().head(10))

print('\nFailed rows — what rules were involved?')
print(execs[~execs['api_success']].groupby('rule_id').size())

API success breakdown:
api_success
True     164
False     50
Name: count, dtype: int64

Failed response samples:
response
{"status": "error", "type": "OAuthException", "code": 190, "message": "access token invalidated"}     27
"No budget to change"                                                                                 20
{"status": "error", "type": "OAuthException", "code": null, "message": "access token invalidated"}     3
Name: count, dtype: int64

Failed rows — what rules were involved?
rule_id
R02    17
R03    19
R05     1
R08     2
R09     8
R12     3
dtype: int64


In [15]:
# --- Inspect duplicate firings ---
# Rules fire every 30 min. An adset that stays in-condition gets fired on repeatedly.
# For impact analysis we want unique "first fire" events per adset per day per rule.

# Count how many times the same adset+rule fired on the same day
dup_counts = (
    execs[execs['api_success']]
    .groupby(['adset_id', 'action_date', 'rule_id'])
    .size()
    .rename('fires')
    .reset_index()
)
print('Distribution of fires per adset+date+rule (successful only):')
print(dup_counts['fires'].value_counts().sort_index())

print('\nWorst repeat-fire cases (same adset+rule+day):')
print(dup_counts[dup_counts['fires'] > 2].sort_values('fires', ascending=False).head(10))

Distribution of fires per adset+date+rule (successful only):
fires
1    27
2    22
3    31
Name: count, dtype: int64

Worst repeat-fire cases (same adset+rule+day):
          adset_id action_date rule_id  fires
0   31107075634783  2026-06-10     R04      3
41  31299219133797  2026-06-06     R04      3
76  31887606876383  2026-06-10     R04      3
74  31865846984262  2026-06-06     R04      3
60  31515347886140  2026-06-07     R04      3
59  31500462861313  2026-06-07     R04      3
58  31469116854677  2026-06-07     R04      3
57  31445874995136  2026-06-07     R04      3
56  31442020869055  2026-06-07     R04      3
54  31327473700930  2026-06-07     R04      3


In [16]:
# --- CLEAN: rule_executions ---
#
# Decision 1: Add api_success flag (already computed above).
#
# Decision 2: Parse action_time to datetime (UTC).
#
# Decision 3: Keep adset_id as string.
#
# Decision 4: Create a deduplicated view (first successful fire per adset+date+rule).
#   Reason: for impact analysis, what matters is WHEN a rule first took effect,
#   not how many times the rule re-evaluated the same condition.
#   We keep the FULL table too for completeness.
#
# Decision 5: Add 'action_type' column: 'Turn OFF' vs 'Budget Decrease' vs 'Turn ON'
#   Reason: simplifies downstream groupby logic.

execs_clean = execs.copy()
execs_clean['adset_id'] = execs_clean['adset_id'].astype(str)
execs_clean['campaign_id'] = execs_clean['campaign_id'].astype(str)
execs_clean['action_time'] = pd.to_datetime(execs_clean['action_time'], utc=True, errors='coerce')
execs_clean['action_date'] = pd.to_datetime(execs_clean['action_date'])

# Derive high-level action type from action_name
def classify_action(name):
    if pd.isna(name):
        return 'unknown'
    n = str(name).upper()
    if 'TURN OFF' in n or 'TURN_OFF' in n:
        return 'Turn OFF'
    if 'TURN ON' in n or 'TURN_ON' in n:
        return 'Turn ON'
    if 'DECREASE' in n:
        return 'Budget Decrease'
    return 'other'

execs_clean['action_type'] = execs_clean['action_name'].apply(classify_action)

# Deduplicated: first successful fire per adset+date+rule
execs_dedup = (
    execs_clean[execs_clean['api_success']]
    .sort_values('action_time')
    .drop_duplicates(subset=['adset_id', 'action_date', 'rule_id'], keep='first')
)

print('Full cleaned execs:', execs_clean.shape)
print('Deduplicated (first-fire-per-adset+date+rule, successes only):', execs_dedup.shape)
print('\nAction type breakdown (deduped):')
print(execs_dedup['action_type'].value_counts())
print('\nRule breakdown (deduped):')
print(execs_dedup['rule_id'].value_counts().sort_index())

Full cleaned execs: (214, 29)
Deduplicated (first-fire-per-adset+date+rule, successes only): (80, 29)

Action type breakdown (deduped):
action_type
Turn OFF           68
Budget Decrease    12
Name: count, dtype: int64

Rule breakdown (deduped):
rule_id
R01     3
R02     6
R03    10
R04    39
R05     5
R06     1
R07     1
R08     9
R10     2
R11     1
R12     3
Name: count, dtype: int64


---
## 5. `buyer_actions.csv`
Manual buyer actions. Concerns: campaign-level rows (null adset_id), mixed event types, and the same dual ID format.

In [17]:
# --- Load and inspect ---
buyers = pd.read_csv(os.path.join(CSV_DIR, 'buyer_actions.csv'))

print('Shape:', buyers.shape)
print('Date range:', buyers['action_time'].min(), '->', buyers['action_time'].max())
print('\nObject type breakdown:')
print(buyers['object_type'].value_counts())
print('\nEvent type breakdown:')
print(buyers['event_type'].value_counts())
print('\nNull counts:')
print(buyers.isnull().sum())
print('\nRows with a note (non-null):', buyers['note'].notna().sum())
print('Notes:')
print(buyers[buyers['note'].notna()]['note'].value_counts())

Shape: (1001, 7)
Date range: 2026-06-06 14:00:10.726391+00 -> 2026-06-12 18:16:27.133162+00

Object type breakdown:
object_type
adset       715
campaign    286
Name: count, dtype: int64

Event type breakdown:
event_type
ui_update_budget              481
ui_adjust_budget              446
update_ad_set_budget           48
update_campaign_budget         25
update_ad_set_bid_strategy      1
Name: count, dtype: int64

Null counts:
action_time      0
object_type      0
adset_id       286
event_type       0
old_budget       1
new_budget       1
note           977
dtype: int64

Rows with a note (non-null): 24
Notes:
note
scaling winner                                                                                   3
budget reset after midnight spike                                                                3
weekend boost                                                                                    2
green 2 days, +30%                                                                

In [18]:
# --- Inspect campaign vs adset level actions ---
print('Rows with null adset_id (campaign-level):', buyers['adset_id'].isnull().sum())
print('Rows with adset_id present:', buyers['adset_id'].notna().sum())

# ID format check on adset-level rows
adset_buyers = buyers[buyers['adset_id'].notna()].copy()
adset_buyers['adset_id_str'] = adset_buyers['adset_id'].astype(str)
adset_buyers['id_len'] = adset_buyers['adset_id_str'].str.len()
print('\nAdset ID length distribution in buyer_actions:')
print(adset_buyers['id_len'].value_counts())

# Budget change direction
buyers['budget_change'] = buyers['new_budget'] - buyers['old_budget']
buyers['budget_direction'] = buyers['budget_change'].apply(
    lambda x: 'increase' if x > 0 else ('decrease' if x < 0 else 'no_change')
)
print('\nBudget change direction:')
print(buyers['budget_direction'].value_counts())

Rows with null adset_id (campaign-level): 286
Rows with adset_id present: 715

Adset ID length distribution in buyer_actions:
id_len
21    569
16     84
20     52
19     10
Name: count, dtype: int64

Budget change direction:
budget_direction
increase     519
decrease     462
no_change     20
Name: count, dtype: int64


In [19]:
# --- CLEAN: buyer_actions ---
#
# Decision 1: Split into adset-level and campaign-level tables.
#   Reason: campaign-level actions don't map to a single adset, so joining them
#   to adset data is ambiguous. We'll keep them separately for reference.
#
# Decision 2: Parse action_time to datetime (UTC).
#
# Decision 3: Keep adset_id as string.
#
# Decision 4: Keep all event_type variants — they all represent budget changes.
#   The different event types appear to be UI source differences, not semantic ones.

buyers_clean = buyers.copy()
buyers_clean['action_time'] = pd.to_datetime(buyers_clean['action_time'], utc=True, errors='coerce')
buyers_clean['action_date'] = buyers_clean['action_time'].dt.normalize()
buyers_clean['adset_id'] = buyers_clean['adset_id'].astype(str).replace('nan', np.nan)

# Split
buyers_adset = buyers_clean[buyers_clean['adset_id'].notna()].copy()
buyers_campaign = buyers_clean[buyers_clean['adset_id'].isna()].copy()

print('Adset-level buyer actions:', len(buyers_adset))
print('Campaign-level buyer actions (excluded from adset joins):', len(buyers_campaign))
print('\nRemaining nulls in adset-level table:')
print(buyers_adset.isnull().sum())

Adset-level buyer actions: 715
Campaign-level buyer actions (excluded from adset joins): 286

Remaining nulls in adset-level table:
action_time          49
object_type           0
adset_id              0
event_type            0
old_budget            1
new_budget            1
note                700
budget_change         1
budget_direction      0
action_date          49
dtype: int64


---
## 6. Cross-Table ID Format Analysis
Key question: can we reliably join `rule_executions` to `daily_adset_performance`? The two ID formats must be understood before any impact analysis.

In [20]:
# --- Which accounts appear in rule_executions? ---
# rule_executions has account_id (numeric). Map to account_name via performance table.
account_map = perf_clean[['fb_ad_account_id', 'account_name']].drop_duplicates()
account_map['fb_ad_account_id'] = account_map['fb_ad_account_id'].astype(str)
print('Account ID -> Name mapping:')
print(account_map.sort_values('account_name').to_string(index=False))

execs_clean['account_id'] = execs_clean['account_id'].astype(str)
execs_with_name = execs_clean.merge(account_map, left_on='account_id', right_on='fb_ad_account_id', how='left')
print('\nRule executions by account:')
print(execs_with_name.groupby('account_name').size().rename('executions').to_string())

Account ID -> Name mapping:
fb_ad_account_id account_name
9948963373021856       ACC-01
9921978465055350       ACC-02
9958842819785590       ACC-03
9913405683583663       ACC-04
 991872650106732       ACC-05
9931629033733169       ACC-06

Rule executions by account:
account_name
ACC-04    214


In [21]:
# --- Try joining rule_executions to performance ---
# Both should have adset_id as string now. Test the join.

test_join = execs_dedup.merge(
    perf_clean[['adset_id', 'date', 'spend', 'roi', 'profit']],
    left_on=['adset_id', 'action_date'],
    right_on=['adset_id', 'date'],
    how='left'
)

matched = test_join['spend'].notna().sum()
total = len(test_join)
print(f'Rule executions that matched a performance row: {matched}/{total} ({100*matched/total:.1f}%)')
print('Unmatched adset_ids from rule_executions (sample):')
unmatched = test_join[test_join['spend'].isna()]['adset_id'].unique()
print(unmatched[:10])
print('\nDo those IDs exist in performance at all (any date)?')
print('Present in perf:', sum(uid in perf_clean['adset_id'].values for uid in unmatched[:10]))

Rule executions that matched a performance row: 80/80 (100.0%)
Unmatched adset_ids from rule_executions (sample):
[]

Do those IDs exist in performance at all (any date)?
Present in perf: 0


In [22]:
# --- Join buyer_actions to performance ---
# Normalize buyer action date to date-only for joining.
buyers_adset['action_date_only'] = buyers_adset['action_time'].dt.date.astype(str)
perf_clean['date_str'] = perf_clean['date'].dt.date.astype(str)

test_buyer_join = buyers_adset.merge(
    perf_clean[['adset_id', 'date_str', 'spend', 'roi']],
    left_on=['adset_id', 'action_date_only'],
    right_on=['adset_id', 'date_str'],
    how='left'
)

matched_b = test_buyer_join['spend'].notna().sum()
total_b = len(test_buyer_join)
print(f'Buyer actions matched to performance: {matched_b}/{total_b} ({100*matched_b/total_b:.1f}%)')
print('Unmatched buyer adset_ids (sample):')
unmatched_b = test_buyer_join[test_buyer_join['spend'].isna()]['adset_id'].unique()
print(unmatched_b[:10])

Buyer actions matched to performance: 0/715 (0.0%)
Unmatched buyer adset_ids (sample):
['7.301227093578122e+17' '7.301338399380581e+17' '31190610559771.0'
 '7.301244348590644e+17' '31273956645572.0' '7.3011790031754e+17'
 '7.301808466254755e+17' '7.301458189283468e+17' '7.301261051349994e+17'
 '7.301269347183974e+17']


---
## 7. Summary
Final state of all cleaned datasets, ready for Task A analysis.

In [23]:
# --- Final summary printout ---
print('='*60)
print('CLEANED DATASET SUMMARY')
print('='*60)

datasets = {
    'perf_clean': perf_clean,
    'meta_clean': meta_clean,
    'rules_clean': rules_clean,
    'execs_clean': execs_clean,
    'execs_dedup': execs_dedup,
    'buyers_adset': buyers_adset,
    'buyers_campaign': buyers_campaign,
}
for name, df in datasets.items():
    print(f'{name:25s}: {df.shape[0]:>6,} rows x {df.shape[1]:>3} cols')

print('\nData quality flags to carry into analysis:')
print(f'  - Zero-spend rows in perf:         {perf_clean["is_zero_spend"].sum():,}')
print(f'  - Failed rule API calls:           {(~execs_clean["api_success"]).sum():,}')
print(f'  - Duplicate rule firings removed:  {len(execs_clean[execs_clean["api_success"]]) - len(execs_dedup):,}')
print(f'  - Campaign-level buyer actions:    {len(buyers_campaign):,}')

CLEANED DATASET SUMMARY
perf_clean               :  4,947 rows x  20 cols
meta_clean               :  7,129 rows x  18 cols
rules_clean              :     12 rows x   6 cols
execs_clean              :    214 rows x  29 cols
execs_dedup              :     80 rows x  29 cols
buyers_adset             :    715 rows x  11 cols
buyers_campaign          :    286 rows x  10 cols

Data quality flags to carry into analysis:
  - Zero-spend rows in perf:         3,000
  - Failed rule API calls:           50
  - Duplicate rule firings removed:  84
  - Campaign-level buyer actions:    286
